# 00 - Load and clean the data
**What this notebook does:** reads the 9 Excel files from `DWC26/raw`, cleans them, and saves fast copies in `DWC26/processed`. All later notebooks read those fast copies.



In [ ]:
# SETUP - run this cell first, every time you open the notebook.
# It connects Google Drive, installs one extra package and loads our helper code.
import os, sys, subprocess, importlib
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/DWC26'          # the shared team folder
except ImportError:                                 # (only used when running outside Colab)
    BASE = os.environ.get('DWC_BASE', 'DWC26')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'python-calamine'], check=False, capture_output=True)
sys.path.insert(0, f'{BASE}/notebooks')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import dwc_utils as du; importlib.reload(du)
P = du.get_paths(BASE)
pd.set_option('display.float_format', lambda x: f'{x:,.4g}')
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.3})
print('Ready. Project folder:', BASE)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Ready. Project folder: /content/drive/MyDrive/DWC26


## Step 1 - Find the 9 files
The notebook finds the files by their number (01 ... 09), so small differences in the names don't matter. You should see 9 lines.

In [ ]:
FILES = du.find_raw_files(P['RAW'])
for num, path in FILES.items():
    print(num, '->', os.path.basename(path), f'({os.path.getsize(path)/1e6:,.0f} MB)')

01 -> 01 Introduction.xlsx (0 MB)
02 -> 02 Field Production.xlsx (16 MB)
03 -> 03 Train Cases 1.xlsx (331 MB)
04 -> 04 Train Cases 2.xlsx (331 MB)
05 -> 05 Train Cases 3.xlsx (331 MB)
06 -> 06 Train Cases 4.xlsx (166 MB)
07 -> 07  Validation Cases.xlsx (248 MB)
08 -> 08 Test Cases.xlsx (248 MB)
09 -> 09 Template Deliverable.xlsx (0 MB)


## Step 2 - The 4 uncertain parameters for each case (file 01)
Every case is one simulator run with its own values of the 4 parameters. You should see **100 cases**. `split` tells which group each case belongs to: train (1-70), validation (71-85), test (86-100).

In [ ]:
params = du.load_parameters(FILES['01'])
params.to_parquet(f"{P['PROC']}/params.parquet")
display(params.head())
print('Value ranges:')
display(params[du.PARAMS].describe().loc[['min', 'mean', 'max']])

  parameter table found in sheet 'Uncertainty Values': 100 cases


,case_id,fault_trans,poro_mult,perm_mult,aquifer_pv,split
0,1,0.05235,1.082,3.047,68.35,train
1,2,0.08946,1.229,6.354,120,train
2,3,0.128,0.8858,1.481,199,train
3,4,0.1455,1.364,8.602,158.1,train
4,5,0.1112,0.9494,5.398,136.2,train


Value ranges:


,fault_trans,poro_mult,perm_mult,aquifer_pv
min,0.05011,0.8012,0.5205,53.95
mean,0.09956,1.148,5.2,125.2
max,0.15,1.494,9.785,199.8


## Step 3 - Production curves (file 02)
File 02 has one sheet per variable (oil / gas / water, cumulative / rate). In each sheet, columns named `Case_1 ... Case_100` are simulator results, and the other columns are the **observed history** of the real field.



In [ ]:
sim, obs = du.load_production(FILES['02'])
sim.to_parquet(f"{P['PROC']}/sim_long.parquet")
obs.to_parquet(f"{P['PROC']}/obs_long.parquet")
print('\nSIMULATED cases:')
display(sim.groupby('var').agg(cases=('case_id', 'nunique'), first_date=('date', 'min'), last_date=('date', 'max')))
print('OBSERVED (real field):')
display(obs.groupby('var').agg(points=('value', 'size'), first_date=('date', 'min'), last_date=('date', 'max')))
missing = [v for v in du.CUMS if v not in set(sim['var'])]
print('All 3 cumulative variables found.' if not missing else f'WARNING: missing {missing} - tell the team lead')

  sheet 'Plots': skipped (empty, or no Date column found)
  sheet 'Field Production': 0 simulated cases, 4 observed columns, variables ['gas_rate', 'oil_rate', 'water_rate']
  sheet 'Gas production cumulative': 100 simulated cases, 1 observed columns, variables ['gas_cum']
  sheet 'Gas production rate': 100 simulated cases, 1 observed columns, variables ['gas_rate']
  sheet 'Oil production cumulative': 100 simulated cases, 1 observed columns, variables ['oil_cum']
  sheet 'Oil production rate': 100 simulated cases, 1 observed columns, variables ['oil_rate']
  sheet 'Water production cumulative': 100 simulated cases, 1 observed columns, variables ['water_cum']
  sheet 'Water production rate': 100 simulated cases, 1 observed columns, variables ['water_rate']

SIMULATED cases:


,cases,first_date,last_date
var,,,
gas_cum,100,1998-01-01,2008-01-01
gas_rate,100,1998-01-01,2008-01-01
oil_cum,100,1998-01-01,2008-01-01
oil_rate,100,1998-01-01,2008-01-01
water_cum,100,1998-01-01,2008-01-01
water_rate,100,1998-01-01,2008-01-01


OBSERVED (real field):


,points,first_date,last_date
var,,,
gas_cum,41,1998-01-01,2008-01-01
gas_rate,41,1998-01-01,2008-01-01
oil_cum,41,1998-01-01,2008-01-01
oil_rate,41,1998-01-01,2008-01-01
water_cum,41,1998-01-01,2008-01-01
water_rate,41,1998-01-01,2008-01-01


All 3 cumulative variables found.


## Step 4 - The grid files (files 03 to 08) - slow, about 30-90 minutes
Each case has a table of thousands of grid cells (small boxes of rock) with their properties. We don't feed millions of cells into the model; we summarise each case into a few numbers (total pore volume, oil-filled pore volume, average porosity and permeability). We also keep the full table for cases 1 and 2, to draw maps and run two checks in notebook 01.



In [ ]:
for key in ['03', '04', '05', '06', '07', '08']:
    if key in FILES:
        print('\n===', os.path.basename(FILES[key]))
        du.convert_grid_file(FILES[key], P['PROC'], keep_full=(1, 2))


=== 03 Train Cases 1.xlsx
  case   1: already done, skipped
  case   2: already done, skipped
  case   3: already done, skipped
  case   4: already done, skipped
  case   5: already done, skipped
  case   6: already done, skipped
  case   7: already done, skipped
  case   8: already done, skipped
  case   9: already done, skipped
  case  10: already done, skipped
  case  11: already done, skipped
  case  12: already done, skipped
  case  13: already done, skipped
  case  14: already done, skipped
  case  15: already done, skipped
  case  16: already done, skipped
  case  17: already done, skipped
  case  18: already done, skipped
  case  19: already done, skipped
  case  20: already done, skipped

=== 04 Train Cases 2.xlsx
  case  21: already done, skipped
  case  22: already done, skipped
  case  23: already done, skipped
  case  24: already done, skipped
  case  25: already done, skipped
  case  26: already done, skipped
  case  27: already done, skipped
  case  28: already done, sk

In [ ]:
feats = du.collect_grid_features(P['PROC'])
feats.to_parquet(f"{P['PROC']}/grid_features.parquet")
print(f'Grid summaries for {len(feats)} cases (should be 100):')
display(feats.head())

Grid summaries for 100 cases (should be 100):


,case_id,n_active_cells,pore_volume_total,oil_pore_volume,gas_pore_volume,water_pore_volume,porosity_mean,permx_geomean,pressure_mean,soil_mean,swat_mean
0,1,42512,4.251e+09,1.401e+08,0,4.111e+09,0.1432,15.05,"4,507",0.1987,0.8013
1,2,42512,7.874e+09,1.591e+08,0,7.715e+09,0.1639,18.39,"4,507",0.1987,0.8013
2,3,42512,9.029e+09,1.146e+08,0,8.914e+09,0.1186,12.37,"4,507",0.1987,0.8013
3,4,42512,1.123e+10,1.765e+08,0,1.106e+10,0.1823,19.97,"4,507",0.1987,0.8013
4,5,42512,6.817e+09,1.229e+08,0,6.694e+09,0.1267,17.59,"4,507",0.1987,0.8013


## Done
`DWC26/processed` now contains `params.parquet`, `sim_long.parquet`, `obs_long.parquet`, `grid_features.parquet` and two folders. **Next: notebook 01.**